# Cryogenic — Offline GPU Music Visualizer

Renders a polished 1080p60 music video from your stems-analysis JSON and the original mp3, using Colab's GPU.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine; L4 is faster).

Workflow:
1. Run the **Setup** cell once (~1 minute).
2. Run the **Upload** cell and drop in your `viz_data_v2.json` and the matching `.mp3`.
3. Run **Render**. ~10–25 min for a 3-min song at 1080p60 on a T4.
4. Preview / download `cryogenic.mp4` from the last cell.

## 1. Setup — install GPU OpenGL + ffmpeg + clone repo

In [ ]:
%%bash
set -e
# Headless EGL + GL libs (Colab GPU runtime already has NVIDIA drivers)
apt-get -qq update
apt-get -qq install -y libegl1 libgles2 libglvnd0 libgl1 libglib2.0-0 ffmpeg >/dev/null
pip -q install moderngl==5.10.0 numpy

# Pull the renderer source.
if [ ! -d /content/music-viz ]; then
  git clone -q https://github.com/splice11/music-viz.git /content/music-viz
fi
cd /content/music-viz
git fetch -q origin
git checkout -q claude/enhance-music-visualization-BB1Ui
git pull -q origin claude/enhance-music-visualization-BB1Ui

echo 'Setup complete.'

In [ ]:
### Verify we'll render on the NVIDIA GPU, not Mesa software (llvmpipe).
### If GL_RENDERER below contains 'NVIDIA' / 'Tesla' / 'L4' you're good.
### If it says 'llvmpipe', stop — that's CPU rendering at ~1 fps.
import os, sys, subprocess, glob
sys.path.insert(0, '/content/music-viz')

print('--- nvidia-smi ---')
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip() or '(no GPU visible)')

print('\n--- EGL ICDs present ---')
for p in sorted(glob.glob('/usr/share/glvnd/egl_vendor.d/*.json')):
    print(' ', p)

# Force libEGL to load only NVIDIA's ICD before moderngl imports it.
nv = [p for p in glob.glob('/usr/share/glvnd/egl_vendor.d/*nvidia*.json')]
if nv:
    os.environ['__EGL_VENDOR_LIBRARY_FILENAMES'] = ':'.join(nv)
os.environ.setdefault('__GLX_VENDOR_LIBRARY_NAME', 'nvidia')

# Drop any cached modules so the env vars above take effect.
for m in list(sys.modules):
    if m.startswith('moderngl') or m.startswith('colab_render'):
        del sys.modules[m]

from colab_render.renderer import _make_ctx
ctx = _make_ctx(verbose=True)
print('\nGL_RENDERER:', ctx.info.get('GL_RENDERER'))
print('GL_VERSION :', ctx.info.get('GL_VERSION'))
assert 'llvmpipe' not in ctx.info.get('GL_RENDERER','').lower(), \
    'Software rendering selected — see EGL ICD list above.'
ctx.release()
print('\nReady to render on GPU.')

## 2. Upload your song + analysis JSON

Drop in **both** files: the `.mp3` and the matching `viz_data_v2.json`.

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('/content/inputs', exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'/content/inputs/{name}')
    print('saved →', f'/content/inputs/{name}')

In [ ]:
import os, glob
mp3s = sorted(glob.glob('/content/inputs/*.mp3') + glob.glob('/content/inputs/*.wav') + glob.glob('/content/inputs/*.m4a'))
jsons = sorted(glob.glob('/content/inputs/*.json'))
assert mp3s,  'No audio file found in /content/inputs (mp3/wav/m4a).'
assert jsons, 'No JSON file found in /content/inputs.'
AUDIO_PATH = mp3s[0]
JSON_PATH  = jsons[0]
print('Audio:', AUDIO_PATH)
print('JSON :', JSON_PATH)

## 3. Render — this is the long step

Defaults: 1920×1080 @ 60 fps, CRF 17 (visually lossless-ish). Drop to 1280×720 for a quick preview.

In [ ]:
WIDTH       = 1920
HEIGHT      = 1080
TARGET_FPS  = 60
OUTPUT_PATH = '/content/cryogenic.mp4'

import importlib, sys
for m in list(sys.modules):
    if m.startswith('colab_render'):
        del sys.modules[m]
import colab_render

print('Building feature timeline…')
fd = colab_render.build(JSON_PATH, target_fps=TARGET_FPS)
print(f'  {fd.n_frames} frames @ {fd.fps:.0f} fps  ({fd.duration:.1f} s)')

print('Rendering…')
colab_render.render_to_video(
    fd,
    audio_path=AUDIO_PATH,
    out_path=OUTPUT_PATH,
    width=WIDTH, height=HEIGHT,
    crf=17, preset='medium',
    progress_every=120,
)

## 4. Preview + download

In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = open(OUTPUT_PATH, 'rb').read()
src = 'data:video/mp4;base64,' + b64encode(data).decode()
HTML(f'<video width=720 controls src="{src}"></video>')

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)